In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/anomaly_labelled_data.csv", parse_dates=['date'])

In [3]:
df.columns

Index(['date', 'temperature', 'humidity', 'pressure', 'label', 'hour', 'month',
       'cos_hour', 'sin_hour', 'cos_month', 'sin_month', 'temp_gradient',
       'humid_gradient', 'press_gradient'],
      dtype='str')

In [4]:
label_map = {}
for i in range(len(df['label'].unique())):
    label_map[df['label'].unique()[i]] = label_map.get(df['label'].unique()[i], i)

df['label'] = df['label'].map(label_map)
df['label'].unique()

array([0, 1, 2, 3])

In [5]:
X = df[['temperature', 'humidity', 'pressure', 'hour', 'month',
       'cos_hour', 'sin_hour', 'cos_month', 'sin_month', 'temp_gradient',
       'humid_gradient', 'press_gradient']]
y = df[['label']]

In [6]:
split_index = int(len(df)*0.80)

In [7]:
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [8]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(56120, 12)
(14031, 12)
(56120, 1)
(14031, 1)


In [9]:
import numpy as np
np.unique(y_train)

array([0, 1, 2, 3])

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

I0000 00:00:1788856790.082153    3446 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788856790.291089    3446 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1788856791.464438    3446 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [11]:
model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(len(np.unique(y_train)),activation="softmax"),
    ]
)

W0000 00:00:1788856793.574379    3446 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [12]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [15]:
model.fit(X_train, y_train, epochs=15,batch_size=64, shuffle=False, validation_split=0.1)

Epoch 1/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 757us/step - accuracy: 0.9161 - loss: 0.3447 - val_accuracy: 0.9947 - val_loss: 0.1809
Epoch 2/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 717us/step - accuracy: 0.9161 - loss: 0.3461 - val_accuracy: 0.9947 - val_loss: 0.1603
Epoch 3/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 696us/step - accuracy: 0.9192 - loss: 0.3095 - val_accuracy: 0.9947 - val_loss: 0.1378
Epoch 4/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 638us/step - accuracy: 0.9134 - loss: 0.4344 - val_accuracy: 0.9947 - val_loss: 0.1431
Epoch 5/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 687us/step - accuracy: 0.9181 - loss: 0.3044 - val_accuracy: 0.9947 - val_loss: 0.1305
Epoch 6/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 663us/step - accuracy: 0.9201 - loss: 0.2944 - val_accuracy: 0.9947 - val_loss: 0.1257
Epoch 7/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 642us/step - accuracy: 0.9217 - loss: 0.2927 - val_accuracy: 0.9947 - val_loss: 0.1197
Epoch 8/15
790/790 ━━━━━━━━━━━━━━━━━━━━ 1s 642us/step - accuracy: 0.9195 - loss: 0.2949 - 

In [16]:
from sklearn.metrics import classification_report, accuracy_score
possibilities = model.predict(X_test)
y_pred = np.argmax(possibilities, axis=1)
score = accuracy_score(y_pred=y_pred, y_true=y_test)
score

439/439 ━━━━━━━━━━━━━━━━━━━━ 0s 365us/step


0.8305894091654195

In [22]:
df = pd.read_csv("data/data_trail.csv")
df = df.drop(index=0)
df

,date,temperature,humidity,pressure,label,cos_hour,sin_hour,cos_month,sin_month,temp_gradient,humid_gradient,press_gradient,elevation,latitude,longitude
1,2023-12-31 19:30:00+00:00,14.900000,92.832680,1014.076538,genuine,0.258819,-0.965926,1.000000,-2.449294e-16,-0.200001,0.589752,-0.200317,11.0,22.56263,88.36304
2,2023-12-31 20:30:00+00:00,14.900000,93.437454,1013.876770,genuine,0.500000,-0.866025,1.000000,-2.449294e-16,0.000000,0.604774,-0.199768,11.0,22.56263,88.36304
3,2023-12-31 21:30:00+00:00,15.150000,93.752861,1013.977478,genuine,0.707107,-0.707107,1.000000,-2.449294e-16,0.250000,0.315407,0.100708,11.0,22.56263,88.36304
4,2023-12-31 22:30:00+00:00,15.050000,94.663445,1014.476624,genuine,0.866025,-0.500000,1.000000,-2.449294e-16,-0.099999,0.910583,0.499146,11.0,22.56263,88.36304
5,2023-12-31 23:30:00+00:00,14.700000,95.885094,1015.174133,genuine,0.965926,-0.258819,1.000000,-2.449294e-16,-0.350000,1.221649,0.697510,11.0,22.56263,88.36304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8803,2025-01-01 13:30:00+00:00,20.200001,67.099625,1013.700806,genuine,-0.965926,-0.258819,0.866025,5.000000e-01,-0.549999,2.448486,0.696899,11.0,22.56263,88.36304
8804,2025-01-01 14:30:00+00:00,19.500000,70.302605,1014.496643,genuine,-0.866025,-0.500000,0.866025,5.000000e-01,-0.700001,3.202980,0.795837,11.0,22.56263,88.36304
8805,2025-01-01 15:30:00+00:00,18.750000,73.666794,1014.792725,genuine,-0.707107,-0.707107,0.866025,5.000000e-01,-0.750000,3.364189,0.296082,11.0,22.56263,88.36304
8806,2025-01-01 16:30:00+00:00,18.049999,77.471115,1014.889771,genuine,-0.500000,-0.866025,0.866025,5.000000e-01,-0.700001,3.804321,0.097046,11.0,22.56263,88.36304
